# Entrenamiento y validación con Alpha Vantage

Voy a entrenar los modelos utilizando únicamente el conjunto de entrenamiento y los compararé con los datos de validación. La prueba final se mantendrá separada hasta elegir completamente el modelo.

In [1]:
from pathlib import Path

import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Localizo la carpeta principal
ruta_actual = Path.cwd().resolve()

if ruta_actual.name == "notebooks":
    ruta_proyecto = ruta_actual.parent
else:
    ruta_proyecto = ruta_actual

archivo_particiones = (
    ruta_proyecto
    / "data"
    / "processed"
    / "eurusd_alpha_vantage_particiones.csv"
)

datos = pd.read_csv(
    archivo_particiones,
    parse_dates=["Date"],
    index_col="Date"
)

datos["Objetivo"] = datos["Objetivo"].astype("Int64")

print("Dimensiones:", datos.shape)

Dimensiones: (4980, 21)


In [2]:
# Variables que utilizarán los modelos
variables_predictoras = [
    "Retorno_diario",
    "Retorno_lag_1",
    "Retorno_lag_2",
    "Retorno_lag_3",
    "Retorno_lag_5",
    "Rango_diario",
    "Cuerpo_vela",
    "Posicion_cierre",
    "Distancia_MA5",
    "Distancia_MA10",
    "Distancia_MA20",
    "Volatilidad_5",
    "Volatilidad_20",
    "RSI_14",
    "MACD_hist"
]

# Por ahora solo utilizo entrenamiento y validación
entrenamiento = datos[
    datos["Particion"] == "entrenamiento"
].copy()

validacion = datos[
    datos["Particion"] == "validacion"
].copy()

X_entrenamiento = entrenamiento[variables_predictoras]
y_entrenamiento = entrenamiento["Objetivo"].astype(int)

X_validacion = validacion[variables_predictoras]
y_validacion = validacion["Objetivo"].astype(int)

print("Entrenamiento:", X_entrenamiento.shape)
print("Validación:", X_validacion.shape)

print("\nDistribución del objetivo en entrenamiento:")
print(
    y_entrenamiento
    .value_counts(normalize=True)
    .sort_index()
    .round(4)
)

Entrenamiento: (4052, 15)
Validación: (522, 15)

Distribución del objetivo en entrenamiento:
Objetivo
0    0.4958
1    0.5042
Name: proportion, dtype: float64


In [3]:
# Utilizo la misma función para evaluar todos los modelos
def calcular_metricas(
    nombre,
    particion,
    y_real,
    y_predicho,
    probabilidades=None
):
    resultado = {
        "Modelo": nombre,
        "Particion": particion,
        "Accuracy": accuracy_score(
            y_real,
            y_predicho
        ),
        "Balanced_accuracy": balanced_accuracy_score(
            y_real,
            y_predicho
        ),
        "Precision": precision_score(
            y_real,
            y_predicho,
            zero_division=0
        ),
        "Recall": recall_score(
            y_real,
            y_predicho,
            zero_division=0
        ),
        "F1": f1_score(
            y_real,
            y_predicho,
            zero_division=0
        ),
        "ROC_AUC": None
    }

    if probabilidades is not None:
        resultado["ROC_AUC"] = roc_auc_score(
            y_real,
            probabilidades
        )

    return resultado


resultados = []

## Modelo base: clase mayoritaria

Este modelo siempre predice la clase más frecuente del entrenamiento. Sirve como referencia mínima para comprobar si los demás modelos realmente aportan algo.

In [4]:
modelo_mayoria = DummyClassifier(
    strategy="most_frequent"
)

modelo_mayoria.fit(
    X_entrenamiento,
    y_entrenamiento
)

pred_mayoria_entrenamiento = modelo_mayoria.predict(
    X_entrenamiento
)

pred_mayoria_validacion = modelo_mayoria.predict(
    X_validacion
)

prob_mayoria_entrenamiento = modelo_mayoria.predict_proba(
    X_entrenamiento
)[:, 1]

prob_mayoria_validacion = modelo_mayoria.predict_proba(
    X_validacion
)[:, 1]

resultados.append(
    calcular_metricas(
        "Clase mayoritaria",
        "Entrenamiento",
        y_entrenamiento,
        pred_mayoria_entrenamiento,
        prob_mayoria_entrenamiento
    )
)

resultados.append(
    calcular_metricas(
        "Clase mayoritaria",
        "Validación",
        y_validacion,
        pred_mayoria_validacion,
        prob_mayoria_validacion
    )
)

## Modelo base de persistencia

Este modelo supone que la dirección de la jornada actual continuará en la jornada siguiente. Si el retorno actual es positivo predice una subida, y si no, predice una bajada.

In [5]:
pred_persistencia_entrenamiento = (
    entrenamiento["Retorno_diario"] > 0
).astype(int)

pred_persistencia_validacion = (
    validacion["Retorno_diario"] > 0
).astype(int)

resultados.append(
    calcular_metricas(
        "Persistencia",
        "Entrenamiento",
        y_entrenamiento,
        pred_persistencia_entrenamiento
    )
)

resultados.append(
    calcular_metricas(
        "Persistencia",
        "Validación",
        y_validacion,
        pred_persistencia_validacion
    )
)

## Regresión logística

Este será el primer modelo que utilizará conjuntamente todas las variables predictoras. El escalado se ajustará únicamente con entrenamiento para no utilizar información de validación.

In [6]:
modelo_logistico = Pipeline([
    (
        "escalador",
        StandardScaler()
    ),
    (
        "modelo",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

modelo_logistico.fit(
    X_entrenamiento,
    y_entrenamiento
)

pred_logistica_entrenamiento = modelo_logistico.predict(
    X_entrenamiento
)

pred_logistica_validacion = modelo_logistico.predict(
    X_validacion
)

prob_logistica_entrenamiento = modelo_logistico.predict_proba(
    X_entrenamiento
)[:, 1]

prob_logistica_validacion = modelo_logistico.predict_proba(
    X_validacion
)[:, 1]

resultados.append(
    calcular_metricas(
        "Regresión logística",
        "Entrenamiento",
        y_entrenamiento,
        pred_logistica_entrenamiento,
        prob_logistica_entrenamiento
    )
)

resultados.append(
    calcular_metricas(
        "Regresión logística",
        "Validación",
        y_validacion,
        pred_logistica_validacion,
        prob_logistica_validacion
    )
)

In [7]:
resultados_iniciales = pd.DataFrame(
    resultados
)

columnas_metricas = [
    "Accuracy",
    "Balanced_accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC_AUC"
]

resultados_iniciales[columnas_metricas] = (
    resultados_iniciales[columnas_metricas]
    .round(4)
)

display(
    resultados_iniciales.set_index(
        ["Modelo", "Particion"]
    )
)

Accuracy  Balanced_accuracy  Precision  \
Modelo              Particion                                               
Clase mayoritaria   Entrenamiento    0.5042             0.5000     0.5042   
                    Validación       0.4904             0.5000     0.4904   
Persistencia        Entrenamiento    0.4798             0.4797     0.4841   
                    Validación       0.4962             0.4960     0.4864   
Regresión logística Entrenamiento    0.5225             0.5218     0.5231   
                    Validación       0.5345             0.5367     0.5202   

                                   Recall      F1  ROC_AUC  
Modelo              Particion                               
Clase mayoritaria   Entrenamiento  1.0000  0.6704   0.5000  
                    Validación     1.0000  0.6581   0.5000  
Persistencia        Entrenamiento  0.4841  0.4841      NaN  
                    Validación     0.4883  0.4873      NaN  
Regresión logística Entrenamiento  0.5996  0.5587   0.5363  
                    Validación     0.6523  0.5789   0.5420

## Comparación sin Posicion_cierre

Voy a repetir la regresión logística eliminando `Posicion_cierre` para comprobar si el rendimiento depende principalmente de esta variable.

In [8]:
# Quito Posicion_cierre de las variables
variables_sin_posicion = [
    variable
    for variable in variables_predictoras
    if variable != "Posicion_cierre"
]

X_entrenamiento_sin_posicion = entrenamiento[
    variables_sin_posicion
]

X_validacion_sin_posicion = validacion[
    variables_sin_posicion
]

# Entreno nuevamente el modelo sin Posicion_cierre
modelo_logistico_sin_posicion = Pipeline([
    (
        "escalador",
        StandardScaler()
    ),
    (
        "modelo",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

modelo_logistico_sin_posicion.fit(
    X_entrenamiento_sin_posicion,
    y_entrenamiento
)

pred_sin_posicion_entrenamiento = (
    modelo_logistico_sin_posicion.predict(
        X_entrenamiento_sin_posicion
    )
)

pred_sin_posicion_validacion = (
    modelo_logistico_sin_posicion.predict(
        X_validacion_sin_posicion
    )
)

prob_sin_posicion_entrenamiento = (
    modelo_logistico_sin_posicion.predict_proba(
        X_entrenamiento_sin_posicion
    )[:, 1]
)

prob_sin_posicion_validacion = (
    modelo_logistico_sin_posicion.predict_proba(
        X_validacion_sin_posicion
    )[:, 1]
)

resultados.append(
    calcular_metricas(
        "Logística sin Posicion_cierre",
        "Entrenamiento",
        y_entrenamiento,
        pred_sin_posicion_entrenamiento,
        prob_sin_posicion_entrenamiento
    )
)

resultados.append(
    calcular_metricas(
        "Logística sin Posicion_cierre",
        "Validación",
        y_validacion,
        pred_sin_posicion_validacion,
        prob_sin_posicion_validacion
    )
)

In [9]:
comparacion_logistica = pd.DataFrame(
    resultados
)

comparacion_logistica = comparacion_logistica[
    comparacion_logistica["Modelo"].isin([
        "Regresión logística",
        "Logística sin Posicion_cierre"
    ])
].copy()

comparacion_logistica[columnas_metricas] = (
    comparacion_logistica[columnas_metricas]
    .round(4)
)

display(
    comparacion_logistica.set_index(
        ["Modelo", "Particion"]
    )
)

#Posicion_cierre no está generando el comportamiento artificial observado con Yahoo Finance
#Puede mantenerse como una variable normal en los siguientes modelos

Accuracy  Balanced_accuracy  \
Modelo                        Particion                                    
Regresión logística           Entrenamiento    0.5225             0.5218   
                              Validación       0.5345             0.5367   
Logística sin Posicion_cierre Entrenamiento    0.5242             0.5235   
                              Validación       0.5326             0.5347   

                                             Precision  Recall      F1  \
Modelo                        Particion                                  
Regresión logística           Entrenamiento     0.5231  0.5996  0.5587   
                              Validación        0.5202  0.6523  0.5789   
Logística sin Posicion_cierre Entrenamiento     0.5244  0.6040  0.5614   
                              Validación        0.5188  0.6484  0.5764   

                                             ROC_AUC  
Modelo                        Particion               
Regresión logística           Entrenamiento   0.5363  
                              Validación      0.5420  
Logística sin Posicion_cierre Entrenamiento   0.5362  
                              Validación      0.5429

## Interpretación de los coeficientes

Como las variables fueron estandarizadas, puedo comparar el tamaño de sus coeficientes. Un valor positivo aumenta la probabilidad estimada de subida, mientras que un valor negativo la reduce.

Los coeficientes no representan relaciones causales, pero permiten comprobar qué variables influyen más en las decisiones de la regresión logística.

In [10]:
# Recupero los coeficientes del modelo con todas las variables (quiero ver cual tuvo mas peso)
coeficientes_todas = pd.Series(
    modelo_logistico
    .named_steps["modelo"]
    .coef_[0],
    index=variables_predictoras
)

# Recupero los coeficientes del modelo sin Posicion_cierre
coeficientes_sin_posicion = pd.Series(
    modelo_logistico_sin_posicion
    .named_steps["modelo"]
    .coef_[0],
    index=variables_sin_posicion
)

# Comparo los coeficientes de los dos modelos
comparacion_coeficientes = pd.DataFrame({
    "Coeficiente_con_todas": coeficientes_todas,
    "Coeficiente_sin_Posicion_cierre": coeficientes_sin_posicion
})

# El valor absoluto permite ordenar por importancia sin importar el signo
comparacion_coeficientes["Importancia_absoluta"] = (
    comparacion_coeficientes[
        "Coeficiente_con_todas"
    ].abs()
)

comparacion_coeficientes = (
    comparacion_coeficientes
    .sort_values(
        "Importancia_absoluta",
        ascending=False
    )
)

display(
    comparacion_coeficientes.round(4)
)

# multiplicando el coeficiente por el valor estandarizado de la variable en esa jornada obtengo la contribucion
# Para cada jornada, el modelo calcula:intercepto + contribución de Retorno_diario + contribución de Cuerpo_vela.....
# Esa suma todavía no es la probabilidad. Primero es una puntuación interna
# La regresión logística transforma después esa puntuación en una probabilidad, si supera el 50%, el modelo predice 1 (ESO PARA CADA JORNADA) 

# En el entrenamiento obtengo los coeficientes de las varaibles
# Despues en la validacion utilizo esos coeficientes como constantes x valor de la variable (escalada) = contribucion de la variable en esa jornada



,Coeficiente_con_todas,Coeficiente_sin_Posicion_cierre,Importancia_absoluta
Cuerpo_vela,-0.3462,-0.3536,0.3462
Retorno_diario,0.2536,0.2539,0.2536
Distancia_MA5,0.1146,0.1149,0.1146
RSI_14,0.1111,0.1102,0.1111
Rango_diario,-0.0948,-0.0951,0.0948
Distancia_MA10,-0.0873,-0.0874,0.0873
Retorno_lag_1,-0.0619,-0.0620,0.0619
Retorno_lag_2,-0.0581,-0.0584,0.0581
Volatilidad_5,0.0427,0.0428,0.0427
Distancia_MA20,-0.0387,-0.0382,0.0387


## Estabilidad anual en validación

Voy a revisar los resultados de 2023 y 2024 por separado. Esto permite comprobar si el rendimiento general se mantiene durante ambos años o si depende solamente de un periodo concreto

In [ ]:
# Obtengo la fecha de la jornada que intenta predecir cada fila
fecha_objetivo = (
    datos.index
    .to_series()
    .shift(-1)
)

anio_objetivo_validacion = (
    fecha_objetivo
    .loc[validacion.index]
    .dt.year
)

# Convierto las predicciones en series para mantener las fechas
pred_logistica_validacion_serie = pd.Series(
    pred_logistica_validacion,
    index=validacion.index
)

prob_logistica_validacion_serie = pd.Series(
    prob_logistica_validacion,
    index=validacion.index
)

pred_sin_posicion_validacion_serie = pd.Series(
    pred_sin_posicion_validacion,
    index=validacion.index
)

prob_sin_posicion_validacion_serie = pd.Series(
    prob_sin_posicion_validacion,
    index=validacion.index
)

resultados_anuales = []

# Evalúo cada año de validación por separado
for anio in [2023, 2024]:
    mascara_anio = (
        anio_objetivo_validacion == anio
    )

    y_anio = y_validacion.loc[
        mascara_anio
    ]

    resultado_con_todas = calcular_metricas(
        "Regresión logística",
        str(anio),
        y_anio,
        pred_logistica_validacion_serie.loc[
            mascara_anio
        ],
        prob_logistica_validacion_serie.loc[
            mascara_anio
        ]
    )

    resultado_con_todas["Filas"] = (
        mascara_anio.sum()
    )

    resultados_anuales.append(
        resultado_con_todas
    )

    resultado_sin_posicion = calcular_metricas(
        "Logística sin Posicion_cierre",
        str(anio),
        y_anio,
        pred_sin_posicion_validacion_serie.loc[
            mascara_anio
        ],
        prob_sin_posicion_validacion_serie.loc[
            mascara_anio
        ]
    )

    resultado_sin_posicion["Filas"] = (
        mascara_anio.sum()
    )

    resultados_anuales.append(
        resultado_sin_posicion
    )

tabla_anual = pd.DataFrame(
    resultados_anuales
)

columnas_tabla_anual = [
    "Modelo",
    "Particion",
    "Filas",
    "Accuracy",
    "Balanced_accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC_AUC"
]

tabla_anual = tabla_anual[
    columnas_tabla_anual
]

tabla_anual[columnas_metricas] = (
    tabla_anual[columnas_metricas]
    .round(4)
)

display(
    tabla_anual.set_index(
        ["Modelo", "Particion"]
    )
)

,,Filas,Accuracy,Balanced_accuracy,Precision,Recall,F1,ROC_AUC
Modelo,Particion,,,,,,,
Regresión logística,2023,260,0.5731,0.5710,0.5636,0.7045,0.6263,0.5911
Logística sin Posicion_cierre,2023,260,0.5731,0.5713,0.5652,0.6894,0.6212,0.5945
Regresión logística,2024,262,0.4962,0.5013,0.4744,0.5968,0.5286,0.4953
Logística sin Posicion_cierre,2024,262,0.4924,0.4981,0.4717,0.6048,0.5300,0.4956


## Conclusión de la regresión logística

La regresión logística obtuvo un resultado un poco mejor que los modelos base cuando se evaluó toda la validación. Sin embargo, al revisar cada año por separado, se vio que el comportamiento no fue estable.

En 2023 el modelo sí consiguió resultados por encima del azar, pero en 2024 volvió a quedar prácticamente alrededor del 50 %. También comprobé que eliminar Posicion_cierre casi no cambió las métricas, así que esta variable no está explicando el rendimiento del modelo con los datos de Alpha Vantage.

Por ahora voy a conservar la regresión logística porque es un modelo fácil de interpretar y me sirve como referencia para comparar después los resultados de Random Forest y XGBoost.